In [11]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

analysis_df = pd.read_csv(
    "clean_online_retail.csv",
    parse_dates=["invoicedate"]
)

analysis_df.head()

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancelled,is_return,revenue,year,month,month_name,day,day_name,hour,quarter,week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,False,83.4,2009,12,December,1,Tuesday,7,4,49
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,81.0,2009,12,December,1,Tuesday,7,4,49
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,81.0,2009,12,December,1,Tuesday,7,4,49
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,False,100.8,2009,12,December,1,Tuesday,7,4,49
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,False,30.0,2009,12,December,1,Tuesday,7,4,49


In [32]:
reference_date = analysis_df["invoicedate"].max() + pd.Timedelta(days=1)

reference_date

Timestamp('2011-12-10 12:50:00')

In [33]:
customer_df = (
    analysis_df[["customer_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

rfm

In [34]:
print("Unique customers:", analysis_df["customer_id"].nunique())
print("Length of unique():", len(analysis_df["customer_id"].unique()))
print("Missing Customer IDs:", analysis_df["customer_id"].isna().sum())

Unique customers: 5878
Length of unique(): 5878
Missing Customer IDs: 0


In [35]:
analysis_df.shape

(779425, 19)

In [36]:
recency = (
    analysis_df.groupby("customer_id")["invoicedate"]
    .max()
    .reset_index()
)

recency["recency"] = (
    reference_date - recency["invoicedate"]
).dt.days

customer_df = customer_df.merge(
    recency[["customer_id", "recency"]],
    on="customer_id",
    how="left"
)


customer_df["frequency"] = (
    analysis_df.groupby("customer_id")["invoice"]
    .nunique()
    .values
)


customer_df["monetary"] = (
    analysis_df.groupby("customer_id")["revenue"]
    .sum()
    .values
)

In [37]:
invoice_value = (
    analysis_df.groupby(
        ["customer_id","invoice"]
    )["revenue"]
    .sum()
    .reset_index()
)

customer_df["avg_order_value"] = (
    invoice_value.groupby("customer_id")["revenue"]
    .mean()
    .values
)



customer_df["total_orders"] = (
    analysis_df.groupby("customer_id")["invoice"]
    .nunique()
    .values
)




basket = (
    analysis_df.groupby(
        ["customer_id","invoice"]
    )["quantity"]
    .sum()
    .reset_index()
)

customer_df["avg_basket_size"] = (
    basket.groupby("customer_id")["quantity"]
    .mean()
    .values
)

product

In [38]:
customer_df["total_products"] = (
    analysis_df.groupby("customer_id")["quantity"]
    .sum()
    .values
)


customer_df["unique_products"] = (
    analysis_df.groupby("customer_id")["stockcode"]
    .nunique()
    .values
)



customer_df["product_diversity"] = (
    customer_df["unique_products"]
    /
    customer_df["total_products"]
)

clv

In [39]:
first_purchase = (
    analysis_df.groupby("customer_id")["invoicedate"]
    .min()
)

last_purchase = (
    analysis_df.groupby("customer_id")["invoicedate"]
    .max()
)


customer_df["customer_lifetime_days"] = (
    last_purchase -
    first_purchase
).dt.days.values



purchase_dates = (
    analysis_df.groupby("customer_id")["invoicedate"]
    .apply(lambda x: x.sort_values().unique())
)

intervals = []

for dates in purchase_dates:

    if len(dates) < 2:

        intervals.append(0)

    else:

        diff = np.diff(dates)

        intervals.append(
            np.mean(diff).astype("timedelta64[D]").astype(int)
        )

customer_df["avg_purchase_interval"] = intervals

time

In [40]:
fav_month = (
    analysis_df.groupby(
        ["customer_id","month_name"]
    )
    .size()
    .reset_index(name="count")
)

fav_month = (
    fav_month.sort_values(
        "count",
        ascending=False
    )
    .drop_duplicates("customer_id")
)

customer_df = customer_df.merge(
    fav_month[
        ["customer_id","month_name"]
    ],
    on="customer_id",
    how="left"
)

customer_df.rename(
    columns={
        "month_name":"favorite_month"
    },
    inplace=True
)

In [41]:
fav_day = (
    analysis_df.groupby(
        ["customer_id","day_name"]
    )
    .size()
    .reset_index(name="count")
)

fav_day = (
    fav_day.sort_values(
        "count",
        ascending=False
    )
    .drop_duplicates("customer_id")
)

customer_df = customer_df.merge(
    fav_day[
        ["customer_id","day_name"]
    ],
    on="customer_id",
    how="left"
)

customer_df.rename(
    columns={
        "day_name":"favorite_day"
    },
    inplace=True
)

country

In [42]:
country = (
    analysis_df.groupby("customer_id")["country"]
    .first()
)

customer_df["country"] = country.values

customer value

In [43]:
customer_df["avg_revenue_per_product"] = (
    customer_df["monetary"]
    /
    customer_df["total_products"]
)

In [44]:
customer_df["revenue_per_order"] = (
    customer_df["monetary"]
    /
    customer_df["frequency"]
)

In [45]:
customer_df["products_per_order"] = (
    customer_df["total_products"]
    /
    customer_df["frequency"]
)

In [46]:
customer_df.to_csv(
    "customer_features.csv",
    index=False
)

print("Customer feature dataset created successfully.")

Customer feature dataset created successfully.
